# <center>Laboratorium Wprowadzenie do Baz Danych</center>

## <center>Indeksacja w PostgreSQL</center>

## Informacje wprowadzające

**Indeks** to dodatkowa struktura danych tworzona przez silnik bazy obok tabeli, która przyspiesza wyszukiwanie wierszy według określonej kolumny (lub zestawu kolumn). Analogia: indeks w książce — zamiast przeszukiwać każdą stronę (tzw. *sequential scan*), sprawdzamy indeks i trafiamy bezpośrednio na właściwą stronę.

Bez indeksu PostgreSQL musi przejrzeć **każdy wiersz** tabeli (*sequential scan*). Dla tabeli z milionem rekordów szukanie po jednym polu może oznaczać milion porównań. Z odpowiednim indeksem (B-tree) liczba porównań spada do ~20 (log₂ 1 000 000 ≈ 20).

### Typy indeksów w PostgreSQL

| Typ | Zastosowanie |
|-----|--------------|
| **B-tree** (domyślny) | Porównania `=`, `<`, `>`, `BETWEEN`, `LIKE 'abc%'` |
| **Hash** | Wyłącznie porównania `=`, szybszy od B-tree tylko dla trafień punktowych |
| **GiST** | Typy geometryczne, wyszukiwanie pełnotekstowe, zakresy |
| **GIN** | Tablice, `jsonb`, wyszukiwanie pełnotekstowe (`tsvector`) |
| **BRIN** | Bardzo duże tabele z naturalnie posortowanymi danymi (np. logi z `timestamp`) |

Na tych laboratoriach skupiamy się na **B-tree** — jest to domyślny i najczęściej stosowany typ.

### Kiedy indeks pomaga, a kiedy szkodzi?

Indeks przyspiesza `SELECT` z filtrowaniem, ale **spowalnia** `INSERT`, `UPDATE` i `DELETE`, ponieważ każda modyfikacja musi zaktualizować również strukturę indeksu. Dlatego:
- Twórz indeksy na kolumnach często używanych w `WHERE`, `JOIN ON`, `ORDER BY`.
- Nie indeksuj kolumn o bardzo niskiej selektywności (np. kolumna `active` z wartościami `0`/`1`) — PostgreSQL może zignorować indeks i wybrać *sequential scan* jako szybszą opcję.
- Unikaj zbędnych indeksów w tabelach intensywnie zapisywanych.

## Konfiguracja środowiska i połączenia

W poniższej komórce następuje nawiązanie połączenia z bazą danych przy użyciu biblioteki SQLAlchemy.

In [1]:
import time

import pandas as pd
from sqlalchemy import URL, create_engine, text

# TODO: Uzupełnij dane do połączenia z bazą danych
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="1234",
    host="localhost",
    port=5432,
    database="dvdrental",
)

try:
    engine = create_engine(db_url)
    connection = engine.connect()
    print("Pomyślnie nawiązano połączenie z bazą danych.")
except Exception as e:
    print(f"Błąd podczas łączenia z bazą danych: {e}")

Pomyślnie nawiązano połączenie z bazą danych.


## Zadania wprowadzające

### EXPLAIN ANALYZE — analiza planów zapytań

Polecenie `EXPLAIN ANALYZE` pokazuje, jak PostgreSQL **faktycznie wykonał** zapytanie:
- **Seq Scan** — pełne przeszukiwanie tabeli (każdy wiersz).
- **Index Scan** — przeszukiwanie indeksu, potem odczyt wskazanych stron tabeli.
- **Index Only Scan** — wszystkie potrzebne dane są w indeksie; tabela nie jest czytana.
- **Bitmap Index Scan** — PostgreSQL zbiera pasujące TID-y z indeksu, a potem czyta tabele blokami.

Kluczowe pola w planie:
- `cost=X..Y` — szacowany koszt (jednostki arbitralne; niższy = lepiej).
- `actual time=X..Y` — rzeczywisty czas wykonania w milisekundach.
- `rows=N` — liczba zwróconych wierszy.
- `loops=N` — ile razy węzeł był wykonywany (ważne dla zagnieżdżonych pętli).

**Zadanie 1.** Wykonaj `EXPLAIN ANALYZE` dla zapytania zwracającego filmy z ceną wypożyczenia `rental_rate = 4.99`. Zaobserwuj, jaki typ skanu pojawia się w planie i dlaczego.

In [2]:
sql_query = text("""--sql
    EXPLAIN ANALYZE
    SELECT * 
    FROM film
    WHERE rental_rate=4.99
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,QUERY PLAN
0,Seq Scan on film (cost=0.00..100.50 rows=336 ...
1,Filter: (rental_rate = 4.99)
2,Rows Removed by Filter: 664
3,Buffers: shared read=88
4,Planning:
5,Buffers: shared hit=166 read=20
6,Planning Time: 4.154 ms
7,Execution Time: 3.334 ms


**Zadanie 2.** Utwórz indeks B-tree na kolumnie `film.rental_rate`. Ponownie wykonaj `EXPLAIN ANALYZE` i porównaj plan. Wyjaśnij, dlaczego PostgreSQL może nadal wybrać *Seq Scan* pomimo obecności indeksu.

> **Wskazówka:** Skorzystaj z `CREATE INDEX IF NOT EXISTS <nazwa> ON <tabela>(<kolumna>)`. Używaj nazw indeksów z prefiksem `idx_lab_`, aby łatwo je potem usunąć.

In [3]:
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_lab_5 ON film(rental_rate);"))
sql_query=text("""
    EXPLAIN ANALYZE 
    SELECT * FROM film 
    WHERE rental_rate = 4.99
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,QUERY PLAN
0,Seq Scan on film (cost=0.00..100.50 rows=336 ...
1,Filter: (rental_rate = 4.99)
2,Rows Removed by Filter: 664
3,Buffers: shared hit=88
4,Planning:
5,Buffers: shared hit=6
6,Planning Time: 0.338 ms
7,Execution Time: 0.563 ms


### Indeks B-tree na kolumnie wysokiej selektywności

Selektywność indeksu to stosunek liczby unikalnych wartości do wszystkich wierszy. Im wyższa selektywność (więcej unikalnych wartości), tym bardziej indeks przyspiesza filtrowanie. Kolumna `last_name` w tabeli `customer` ma znacznie wyższą selektywność niż `rental_rate`.

**Zadanie 3.** Sprawdź plan zapytania wyszukującego klientów o nazwisku `'Smith'`. Wykonaj je kilka razy i zanotuj rzeczywisty czas wykonania.

In [4]:
sql_query = text("""
    EXPLAIN ANALYZE 
    SELECT * 
    FROM customer
    WHERE last_name = 'Smith'
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

# Czas wykonania to ok. 0,04 ms

,QUERY PLAN
0,Index Scan using idx_last_name on customer (c...
1,Index Cond: ((last_name)::text = 'Smith'::text)
2,Index Searches: 1
3,Buffers: shared read=3
4,Planning:
5,Buffers: shared hit=89 read=6
6,Planning Time: 2.673 ms
7,Execution Time: 0.895 ms


**Zadanie 4.** Utwórz indeks B-tree na kolumnie `customer.last_name`. Ponownie wykonaj `EXPLAIN ANALYZE` i porównaj plan oraz czas. Czy PostgreSQL używa teraz indeksu?

In [5]:
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_lab_5_2 ON customer(last_name);"))
sql_query = text("""
    EXPLAIN ANALYZE 
    SELECT * 
    FROM customer
    WHERE last_name = 'Smith'
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

# Czas wykonania to ok. 0,03 ms

,QUERY PLAN
0,Index Scan using idx_last_name on customer (c...
1,Index Cond: ((last_name)::text = 'Smith'::text)
2,Index Searches: 1
3,Buffers: shared hit=3
4,Planning:
5,Buffers: shared hit=6
6,Planning Time: 0.146 ms
7,Execution Time: 0.029 ms


### Pomiar wydajności przed i po indeksie

Na małych tabelach różnica w czasie jest praktycznie niezauważalna — PostgreSQL i tak odczytuje wszystko z pamięci podręcznej. Żeby zobaczyć realny efekt, tworzymy dedykowaną tabelę z **10 milionami wierszy** i porównujemy czas zapytania bez indeksu (*Seq Scan*) z czasem po jego założeniu (*Index Scan*).

Dane generujemy po stronie serwera funkcją `generate_series()` — jest to wielokrotnie szybsze niż wysyłanie milionów wierszy przez sterownik Python. Po masowym INSERT konieczny jest `VACUUM ANALYZE`, który odświeża statystyki planisty; bez tego PostgreSQL może podjąć błędną decyzję o planie wykonania.

> **Uwaga:** `VACUUM` nie może działać wewnątrz transakcji — uruchamiamy go w trybie `AUTOCOMMIT`.

**Zadanie 5.** Utwórz tabelę `lab05_benchmark` z 10 milionami wierszy (losowe `customer_id` i `amount`). Zmierz czas zapytania punktowego (`customer_id = 341`) przed i po założeniu indeksu na `customer_id`. Dla obu przypadków wyświetl plan `EXPLAIN ANALYZE` i oblicz uzyskane przyspieszenie. Na koniec usuń tabelę.

> **Wskazówki:**
> - Dane wygeneruj server-side: `INSERT INTO ... SELECT ... FROM generate_series(1, 10000000)`.
> - Po INSERT i po CREATE INDEX uruchom `VACUUM ANALYZE` w trybie `AUTOCOMMIT`: `engine.connect().execution_options(isolation_level="AUTOCOMMIT")`.
> - Seq Scan na 10M wierszach jest wolny — dla pomiaru bez indeksu wystarczą 3 powtórzenia; z indeksem powtórz 20 razy.

In [6]:
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("DROP TABLE IF EXISTS lab05_benchmark;"))
    conn.execute(text("CREATE TABLE lab05_benchmark AS " \
    "SELECT (random() * 1000000)::int as customer_id, (random()*1000)::numeric(10, 2) as amount " \
    "FROM generate_series(1, 1000000);"))
    conn.execute(text("VACUUM ANALYZE lab05_benchmark;"))

sql_no_index = text("EXPLAIN ANALYZE SELECT * FROM lab05_benchmark WHERE customer_id = 341")
df = pd.read_sql(sql_no_index, con=engine)
display(df)

with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("CREATE INDEX idx_lab_5_3 ON lab05_benchmark(customer_id);"))
    conn.execute(text("VACUUM ANALYZE lab05_benchmark;"))

sql_index = text("EXPLAIN ANALYZE SELECT * FROM lab05_benchmark WHERE customer_id = 341")
df = pd.read_sql(sql_index, con=engine)
display(df)

,QUERY PLAN
0,Gather (cost=1000.00..11648.53 rows=2 width=1...
1,Workers Planned: 2
2,Workers Launched: 2
3,Buffers: shared hit=2304 read=3136
4,-> Parallel Seq Scan on lab05_benchmark (c...
5,Filter: (customer_id = 341)
6,Rows Removed by Filter: 333333
7,Buffers: shared hit=2304 read=3136
8,Planning:
9,Buffers: shared hit=13


,QUERY PLAN
0,Index Scan using idx_lab_5_3 on lab05_benchmar...
1,Index Cond: (customer_id = 341)
2,Index Searches: 1
3,Buffers: shared read=3
4,Planning:
5,Buffers: shared hit=9
6,Planning Time: 0.066 ms
7,Execution Time: 0.055 ms


### Indeks unikalny

Indeks unikalny (`UNIQUE INDEX`) pełni podwójną rolę:
1. Przyspiesza wyszukiwanie (jak każdy indeks).
2. Wymusza unikalność wartości w kolumnie — baza odrzuci `INSERT` lub `UPDATE` próbujący wstawić duplikat.

Uwaga: klucze główne (`PRIMARY KEY`) i więzy `UNIQUE` są implementowane jako indeksy unikalne — PostgreSQL tworzy je automatycznie.

**Zadanie 6.** Utwórz unikalny indeks na kolumnie `film.title`. Następnie spróbuj wstawić film z tytułem, który już istnieje w bazie, i zaobserwuj błąd.

> **Wskazówka:** Skorzystaj z `CREATE UNIQUE INDEX IF NOT EXISTS ...`. Przy próbie insertu użyj bloku `try/except` i wydrukuj komunikat błędu.

In [7]:
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("CREATE UNIQUE INDEX IF NOT EXISTS lab_05_4 on film(title)"))
    try:
        conn.execute(text("INSERT INTO film(title, language_id) VALUES ('Academy Dinosaur', 1)"))
    except Exception as e:
        print(e)

(psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "lab_05_4"
DETAIL:  Key (title)=(Academy Dinosaur) already exists.

[SQL: INSERT INTO film(title, language_id) VALUES ('Academy Dinosaur', 1)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


### Indeks częściowy (Partial Index)

Indeks częściowy indeksuje tylko **podzbiór wierszy** spełniający klauzulę `WHERE`. Dzięki temu jest mniejszy i szybszy niż indeks pełny, jeśli zapytania często filtrują po tym samym warunku.

Przykład zastosowania: indeks na wypożyczeniach, które nie zostały jeszcze zwrócone (`return_date IS NULL`). Takich rekordów jest znacznie mniej niż wszystkich wypożyczeń, co sprawia, że indeks jest bardzo selektywny.

**Zadanie 7.** Utwórz indeks częściowy na tabeli `rental` obejmujący tylko wiersze, gdzie `return_date IS NULL` (niezwrócone wypożyczenia). Porównaj plany zapytań: dla niezwróconych vs. dla zwróconych wypożyczeń.

> **Wskazówka:** Składnia indeksu częściowego: `CREATE INDEX ... ON tabela(kolumna) WHERE warunek`.

In [8]:
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS lab_05_5 on rental(return_date) WHERE return_date IS NULL"))

sql_returned = text("EXPLAIN ANALYZE SELECT * FROM rental WHERE return_date IS NOT NULL")
df = pd.read_sql(sql_returned, con=engine)
display(df)

sql_not_returned = text("EXPLAIN ANALYZE SELECT * FROM rental WHERE return_date IS NULL")
df = pd.read_sql(sql_not_returned, con=engine)
display(df)

,QUERY PLAN
0,Seq Scan on rental (cost=0.00..310.44 rows=15...
1,Filter: (return_date IS NOT NULL)
2,Rows Removed by Filter: 183
3,Buffers: shared read=150
4,Planning:
5,Buffers: shared hit=78 read=7
6,Planning Time: 2.941 ms
7,Execution Time: 8.577 ms


,QUERY PLAN
0,Index Scan using lab_05_5 on rental (cost=0.1...
1,Index Searches: 1
2,Buffers: shared hit=42 read=1
3,Planning:
4,Buffers: shared hit=3
5,Planning Time: 0.097 ms
6,Execution Time: 0.139 ms


### Indeks złożony (Composite Index)

Indeks złożony obejmuje **kilka kolumn**. Jest używany wtedy, gdy zapytanie filtruje po wielu kolumnach jednocześnie. Kluczowa zasada: indeks złożony `(A, B)` przyspiesza zapytania filtrujące po `A` lub `(A, B)`, ale **nie** zapytania filtrujące tylko po `B` — kolejność kolumn w definicji indeksu ma znaczenie.

**Zadanie 8.** Utwórz indeks złożony na `payment(customer_id, amount)`. Porównaj plany zapytań dla trzech przypadków: filtrowanie po `customer_id` i `amount`, tylko po `customer_id`, oraz tylko po `amount`.

In [9]:
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS lab_05_6 on payment(customer_id, amount);"))

sql_ab = text("EXPLAIN ANALYZE SELECT * FROM payment WHERE customer_id = 341 AND amount>10")
df = pd.read_sql(sql_ab, con=engine)
display(df)

sql_a = text("EXPLAIN ANALYZE SELECT * FROM payment WHERE customer_id = 341")
df = pd.read_sql(sql_a, con=engine)
display(df)

sql_b = text("EXPLAIN ANALYZE SELECT * FROM payment WHERE amount>10")
df = pd.read_sql(sql_b, con=engine)
display(df)

,QUERY PLAN
0,Bitmap Heap Scan on payment (cost=4.46..61.02...
1,Recheck Cond: (customer_id = 341)
2,Filter: (amount > '10'::numeric)
3,Rows Removed by Filter: 22
4,Heap Blocks: exact=3
5,Buffers: shared hit=6 read=2
6,-> Bitmap Index Scan on idx_fk_customer_id ...
7,Index Cond: (customer_id = 341)
8,Index Searches: 1
9,Buffers: shared hit=3 read=2


,QUERY PLAN
0,Bitmap Heap Scan on payment (cost=4.46..60.97...
1,Recheck Cond: (customer_id = 341)
2,Heap Blocks: exact=3
3,Buffers: shared hit=5
4,-> Bitmap Index Scan on idx_fk_customer_id ...
5,Index Cond: (customer_id = 341)
6,Index Searches: 1
7,Buffers: shared hit=2
8,Planning Time: 0.067 ms
9,Execution Time: 0.039 ms


,QUERY PLAN
0,Seq Scan on payment (cost=0.00..290.45 rows=1...
1,Filter: (amount > '10'::numeric)
2,Rows Removed by Filter: 14489
3,Buffers: shared hit=108
4,Planning Time: 0.071 ms
5,Execution Time: 1.459 ms


### Wpływ indeksów na operacje JOIN

PostgreSQL wybiera strategię wykonania JOIN na podstawie dostępnych indeksów, rozmiarów tabel i statystyk planisty:

| Strategia | Kiedy wybierana | Indeks potrzebny? |
|---|---|---|
| **Hash Join** | Brak indeksu na kluczu złączenia; obie strony duże | Nie |
| **Nested Loop + Index Scan** | Indeks na kluczu złączenia tabeli wewnętrznej; mały wynik | Tak |
| **Merge Join** | Obie strony posortowane (np. przez indeksy na kluczu JOIN) | Opcjonalnie |

W praktyce brak indeksu na kolumnie klucza obcego jest jednym z najczęstszych błędów wydajnościowych. PostgreSQL **nie tworzy automatycznie** indeksów na kolumnach FK — w przeciwieństwie do MySQL. Efekt: JOIN na dużej tabeli faktów bez indeksu na FK wymusza Hash Join i pełny skan tabeli.

**Zadanie 9.** Utwórz tabelę `lab05_fact` (5 mln wierszy z losowymi `customer_id` i `amount`) oraz tabelę `lab05_dim` (599 wierszy). Zmierz czas złączenia tych tabel po `customer_id` przed i po założeniu indeksu na `lab05_fact.customer_id`. Zaobserwuj zmianę strategii JOIN w planach `EXPLAIN ANALYZE` (Hash Join → Nested Loop) i oblicz przyspieszenie. Na koniec usuń obie tabele.

> **Wskazówki:**
> - Użyj `generate_series()` do wypełnienia obu tabel po stronie serwera.
> - Po INSERT i po CREATE INDEX uruchom `VACUUM ANALYZE` w trybie `AUTOCOMMIT`.
> - Dla Seq Scan na 5M wierszach wystarczą 3 powtórzenia; z indeksem powtórz 20 razy.

In [10]:
from sqlalchemy import text

with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("DROP TABLE IF EXISTS lab05_fact;"))
    conn.execute(text("DROP TABLE IF EXISTS lab05_dim;"))
    
    conn.execute(text("""
        CREATE TABLE lab05_fact AS 
        SELECT (random() * 5000000)::int as customer_id, (random() * 1000)::int as amount 
        FROM generate_series(1, 5000000);
    """))
    
    conn.execute(text("""
        CREATE TABLE lab05_dim AS 
        SELECT (random() * 599)::int as customer_id, (random() * 1000)::int as amount 
        FROM generate_series(1, 599);
    """))
    
    conn.execute(text("VACUUM ANALYZE lab05_fact;"))
    conn.execute(text("VACUUM ANALYZE lab05_dim;"))

sql_no_index = text("""
    EXPLAIN ANALYZE 
    SELECT * FROM lab05_fact f 
    JOIN lab05_dim d ON f.customer_id = d.customer_id;
""")
display(pd.read_sql(sql_no_index, con=engine))

with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("CREATE INDEX idx_fact_cust ON lab05_fact(customer_id);"))
    conn.execute(text("VACUUM ANALYZE lab05_fact;"))

sql_with_index = text("""
    EXPLAIN ANALYZE 
    SELECT * FROM lab05_fact f 
    JOIN lab05_dim d ON f.customer_id = d.customer_id;
""")
display(pd.read_sql(sql_with_index, con=engine))

,QUERY PLAN
0,Gather (cost=1016.48..67549.12 rows=1131 widt...
1,Workers Planned: 2
2,Workers Launched: 2
3,Buffers: shared hit=2313 read=19840
4,-> Hash Join (cost=16.48..66436.02 rows=47...
5,Hash Cond: (f.customer_id = d.customer...
6,Buffers: shared hit=2313 read=19840
7,-> Parallel Seq Scan on lab05_fact f ...
8,Buffers: shared hit=2304 read=19840
9,-> Hash (cost=8.99..8.99 rows=599 wi...


,QUERY PLAN
0,Merge Join (cost=37.26..79.68 rows=1185 width...
1,Merge Cond: (f.customer_id = d.customer_id)
2,Buffers: shared hit=107 read=503
3,-> Index Scan using idx_fact_cust on lab05_...
4,Index Searches: 1
5,Buffers: shared hit=104 read=503
6,-> Sort (cost=36.62..38.12 rows=599 width=...
7,Sort Key: d.customer_id
8,Sort Method: quicksort Memory: 39kB
9,Buffers: shared hit=3


### Usuwanie indeksów (DROP INDEX)

Indeksy można usunąć poleceniem `DROP INDEX`. Warto to zrobić po laboratorium, aby nie zaśmiecać środowiska testowego. Po usunięciu plan powróci do *Seq Scan*.

**Zadanie 10.** Usuń wszystkie indeksy utworzone podczas tych laboratoriów. Zweryfikuj, że plan zapytania dla `customer.last_name` wrócił do *Seq Scan*.

In [11]:
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("DROP INDEX IF EXISTS idx_lab_5;"))
    conn.execute(text("DROP INDEX IF EXISTS idx_lab_5_2;"))
    conn.execute(text("DROP INDEX IF EXISTS idx_lab_5_3;"))
    conn.execute(text("DROP INDEX IF EXISTS idx_lab_5_4;"))
    conn.execute(text("DROP INDEX IF EXISTS idx_lab_5_5;"))
    conn.execute(text("DROP INDEX IF EXISTS idx_lab_5_6;"))
    conn.execute(text("DROP INDEX IF EXISTS idx_fact_cust;"))

sql_query = text("EXPLAIN ANALYZE SELECT * FROM customer WHERE last_name = 'Smith' ")
display(pd.read_sql(sql_query, con = engine))

,QUERY PLAN
0,Index Scan using idx_last_name on customer (c...
1,Index Cond: ((last_name)::text = 'Smith'::text)
2,Index Searches: 1
3,Buffers: shared hit=3
4,Planning Time: 0.108 ms
5,Execution Time: 0.053 ms


### Istniejące indeksy w bazie DVDRental

PostgreSQL udostępnia widok systemowy `pg_indexes`, który pozwala sprawdzić, jakie indeksy już istnieją w bazie.

**Zadanie 11.** Wyświetl listę wszystkich indeksów w schemacie `public` bazy DVDRental. Ile ich jest? Na jakich tabelach i kolumnach zostały założone?

In [12]:
sql_query = text("""
    SELECT tablename, indexname, indexdef 
    FROM pg_indexes 
    WHERE schemaname = 'public'
    ORDER BY tablename, indexname;
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,tablename,indexname,indexdef
0,actor,actor_pkey,CREATE UNIQUE INDEX actor_pkey ON public.actor...
1,actor,idx_actor_last_name,CREATE INDEX idx_actor_last_name ON public.act...
2,address,address_pkey,CREATE UNIQUE INDEX address_pkey ON public.add...
3,address,idx_fk_city_id,CREATE INDEX idx_fk_city_id ON public.address ...
4,category,category_pkey,CREATE UNIQUE INDEX category_pkey ON public.ca...
5,city,city_pkey,CREATE UNIQUE INDEX city_pkey ON public.city U...
6,city,idx_fk_country_id,CREATE INDEX idx_fk_country_id ON public.city ...
7,country,country_pkey,CREATE UNIQUE INDEX country_pkey ON public.cou...
8,customer,customer_pkey,CREATE UNIQUE INDEX customer_pkey ON public.cu...
9,customer,idx_fk_address_id,CREATE INDEX idx_fk_address_id ON public.custo...


## Zadanie implementacyjne

Zaimplementuj sparametryzowane zapytania w udostępnionym pliku `main.py`, a następnie zaimportuj je do notatnika i przetestuj wywołania z użyciem różnych parametrów.

In [13]:
from main import (
    active_customers_in_store,
    customer_rentals_in_range,
    customers_by_last_name,
    payments_above_amount,
    unreturned_rentals,
)

[INFO] Successfully connected to the database.


In [14]:
df = customers_by_last_name("Smith")
display(df)

,customer_id,first_name,last_name,email
0,1,Mary,Smith,mary.smith@sakilacustomer.org


In [15]:
df = payments_above_amount(10.99)
display(df)

,first_name,last_name,amount,payment_date
0,Karen,Jackson,11.99,2007-04-29 21:06:07.996577
1,Kent,Arsenault,11.99,2007-04-07 19:14:17.996577
2,Terrance,Roush,11.99,2007-04-06 21:26:57.996577
3,Vanessa,Sims,11.99,2007-03-23 20:47:59.996577
4,Rosemary,Schmidt,11.99,2007-03-22 22:17:22.996577
5,Victoria,Gibson,11.99,2007-03-21 22:02:26.996577
6,Nicholas,Barfield,11.99,2007-03-21 21:57:24.996577
7,Tanya,Gilbert,11.99,2007-03-02 20:46:39.996577


In [16]:
df = active_customers_in_store(1)
display(df)

,customer_id,first_name,last_name,customer_id,email
0,505,Rafael,Abney,505,rafael.abney@sakilacustomer.org
1,504,Nathaniel,Adam,504,nathaniel.adam@sakilacustomer.org
2,96,Diana,Alexander,96,diana.alexander@sakilacustomer.org
3,470,Gordon,Allard,470,gordon.allard@sakilacustomer.org
4,326,Jose,Andrew,326,jose.andrew@sakilacustomer.org
...,...,...,...,...,...
313,78,Lori,Wood,78,lori.wood@sakilacustomer.org
314,107,Florence,Woods,107,florence.woods@sakilacustomer.org
315,318,Brian,Wyman,318,brian.wyman@sakilacustomer.org
316,402,Luis,Yanez,402,luis.yanez@sakilacustomer.org


In [17]:
df = unreturned_rentals()
display(df)

,rental_id,customer_id,title,rental_date
0,14098,554,Academy Dinosaur,2005-08-21 00:30:32
1,11496,155,Hyde Doctor,2006-02-14 15:16:03
2,11541,335,Hunger Roof,2006-02-14 15:16:03
3,11563,83,Frisco Forrest,2006-02-14 15:16:03
4,11577,219,Titans Jerk,2006-02-14 15:16:03
...,...,...,...,...
178,15862,215,Dances None,2006-02-14 15:16:03
179,15867,505,Conversation Downhill,2006-02-14 15:16:03
180,15875,41,Shock Cabin,2006-02-14 15:16:03
181,15894,168,Wedding Apollo,2006-02-14 15:16:03


In [18]:
df = customer_rentals_in_range(1, "2005-05-01", "2005-07-01")
display(df)

,rental_id,title,rental_date,return_date
0,76,Patient Sister,2005-05-25 11:30:37,2005-06-03 12:00:37
1,573,Talented Homicide,2005-05-28 10:35:23,2005-06-03 06:32:23
2,1185,Musketeers Wait,2005-06-15 00:54:12,2005-06-23 02:42:12
3,1422,Detective Vision,2005-06-15 18:02:53,2005-06-19 15:54:53
4,1476,Ferris Mother,2005-06-15 21:08:46,2005-06-25 02:26:46
5,1725,Closer Bang,2005-06-16 15:18:57,2005-06-17 21:05:57
6,2308,Attacks Hate,2005-06-18 08:41:48,2005-06-22 03:36:48
7,2363,Savannah Town,2005-06-18 13:33:59,2005-06-19 17:40:59
8,3284,Youth Kick,2005-06-21 06:24:45,2005-06-28 03:28:45
